# SIS #3 — The Warehouse Sequence Detective
## RNN vs. LSTM: Does the order of events matter?

**CSCI1409 · Deep Learning · Bachelor’s degree · Week 11 · 4 points**
**Format:** In-class team assignment, followed by an individual oral defense
**Team size:** 1–3 students · **Submission:** one completed `.ipynb` per team on Microsoft Teams

### Scenario
A warehouse camera system detects these five actions:

| Event ID | Action |
|:--:|---|
| `0` | IDLE |
| `1` | CHECK |
| `2` | MOVE |
| `3` | SCAN |
| `4` | PACK |

Your task is to classify a sequence of events:

- **Label `1` (correct):** `SCAN` occurred **before** `PACK`.
- **Label `0` (incorrect):** `PACK` occurred **before** `SCAN`.

Every sequence contains **exactly one SCAN and one PACK**, plus unrelated actions. So knowing *which* actions occurred is not enough: your model must learn **their order**.

**Example:** `IDLE → SCAN → MOVE → PACK` is class **1**, whereas `IDLE → PACK → MOVE → SCAN` is class **0**.

### What is provided
The **first code cell below is the entire provided dataset** and the only provided code. Run it once without changing it. It creates the train, validation and test sets in memory, using only NumPy. **No downloads, external files, data cleaning, or GPU are needed.**

The input arrays are `X_train`, `X_val`, `X_test` and the labels are `y_train`, `y_val`, `y_test`. Each input is a sequence of **16 one-hot event vectors** (5 features per time step).

### Your task — write ALL code yourselves
You must implement **every model, training loop, metric, and experiment from scratch**, using the **blank code cells** provided after the dataset. You may use standard PyTorch building blocks such as `nn.Linear`, `nn.RNN`, `nn.LSTM`, `CrossEntropyLoss`, `Adam` and `DataLoader`. You do **not** need to implement LSTM gates manually.

- **A. Explore + pipeline:** Inspect the data, then write reusable training and evaluation functions.
- **B. Baseline:** Train an order-blind model.
- **C. RNN:** Build and train a vanilla RNN classifier.
- **D. LSTM:** Build and train an LSTM classifier.
- **E. Test + reversal:** Evaluate all models on the test set once, then reverse the test sequences and compare predictions.
- **F. Conclusions:** Report your results and answer the short questions.
- **G. Optional challenges** (not graded).

Every model trains in a few seconds on a laptop CPU. Spend your time on **predicting** results before you run, **reading** your metrics, and **explaining** why each model behaves as it does: that is what the questions and the defense are about.

### Reproducibility
- Define `SEED = 0`. Call `torch.manual_seed(SEED)` **immediately before creating each model**, and seed the mini-batch shuffling, e.g. `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(SEED))`.
- The numbers in your Part F table must be the ones your notebook prints after **Kernel → Restart & Run All**.

### Rules and AI policy
This assignment follows the course’s **Level C (AI-Assisted)** policy.

- **Chat-based AI is allowed** for explanations, debugging and individual code snippets. You must **write, assemble, run, verify and understand** your own complete solution.
- **AI coding agents and code-editing assistants are not allowed:** no Codex, Claude Code, IDE-integrated coding agents, or tools that automatically create/edit/run your notebook for you. Use AI **only through a chat interface**.
- Cite any code you adapt from outside sources in a short comment. Verify everything an AI tells you: you will be asked to explain it.
- Do not change the provided dataset, labels or splits. Train on **train only**. In Parts A–D, check your models with the **validation** set only. The **test** set is used **once**, in Part E, after all models are trained.
- **All team members must participate and can be questioned on any part of the code**, not just the part they personally wrote.
- You are graded on correct implementation and understanding, **not** on achieving a particular accuracy threshold. Show your results even if a model performs poorly.

### Grading — 4.0 points

| Part | What you deliver | Points |
|---|---|---:|
| A | Data exploration; reusable training and evaluation functions | **0.50** |
| B | Order-blind baseline, trained and explained (F-Q1) | **0.50** |
| C | Vanilla RNN, trained and explained (F-Q2) | **0.75** |
| D | LSTM, trained and explained (F-Q3, F-Q4) | **0.75** |
| E | Test evaluation, reversal experiment and interpretation (F-Q5) | **0.50** |
| **Oral defense** | Individual understanding of the code and concepts | **1.00** |
| **Total** | | **4.00** |

Each part is graded on correct code, a reported result, and a correct explanation. Part G is optional and not graded.

**Suggested settings** (use the **same** settings for all models): `hidden_size=24`, `batch_size=128`, `epochs=10`, `learning_rate=0.01`, one recurrent layer. Use **2 output logits** for the two classes and cross-entropy loss.

---

### PROVIDED DATASET — run the following cell without editing it


In [ ]:
import numpy as np

EVENTS = ["IDLE", "CHECK", "MOVE", "SCAN", "PACK"]
SEQUENCE_LENGTH = 16 # 

def _make_split(n_samples, seed):
    rng = np.random.default_rng(seed)
    event_ids = rng.integers(0, 3, size=(n_samples, SEQUENCE_LENGTH), dtype=np.uint8)
    labels = np.array([0, 1] * (n_samples // 2), dtype=np.int64)
    rng.shuffle(labels)
    for i in range(n_samples):
        first, second = sorted(rng.choice(SEQUENCE_LENGTH, size=2, replace=False))
        event_ids[i, first], event_ids[i, second] = (
            (3, 4) if labels[i] == 1 else (4, 3)
        )
    return np.eye(5, dtype=np.float32)[event_ids], labels

X_train, y_train = _make_split(1200, seed=4)
X_val, y_val = _make_split(240, seed=5)
X_test, y_test = _make_split(240, seed=6)

print("Dataset ready — zero downloads required")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Dataset ready — zero downloads required
Train: (1200, 16, 5) (1200,)
Val:   (240, 16, 5) (240,)
Test:  (240, 16, 5) (240,)


## A — Explore the data and build the pipeline

### A1. Explore

**Write your own Python code** to:

1. Print the input shape and class counts for the train split.
2. Display **one example with label 1** and **one example with label 0** as readable event names, using the `EVENTS` list.
3. Inspect one time-step vector (for example, the first vector in the first sequence).

**Think about:** What do `N`, `T` and `5` represent in a shape `(N, T, 5)`? Why can two sequences containing the same number of SCAN and PACK events have different labels?

In [3]:
# A1 — Explore
print("Train input shape (N, T, features):", X_train.shape)
vals, counts = np.unique(y_train, return_counts=True)
print("Train class counts:", dict(zip(vals.tolist(), counts.tolist())))

def to_names(x):
    return " → ".join(EVENTS[i] for i in x.argmax(axis=1))

i1 = np.where(y_train == 1)[0][0]
i0 = np.where(y_train == 0)[0][0]
print(f"\nLabel 1 example (#{i1}):\n ", to_names(X_train[i1]))
print(f"Label 0 example (#{i0}):\n ", to_names(X_train[i0]))

print("\nFirst time-step vector of first sequence:", X_train[0, 0], "->", EVENTS[X_train[0, 0].argmax()])

# Sanity: every sequence has exactly one SCAN and one PACK
ids = X_train.argmax(axis=2)
print("Exactly one SCAN & one PACK in all train sequences:",
      bool(((ids == 3).sum(1) == 1).all() and ((ids == 4).sum(1) == 1).all()))

Train input shape (N, T, features): (1200, 16, 5)
Train class counts: {0: 600, 1: 600}

Label 1 example (#0):
  IDLE → IDLE → MOVE → MOVE → MOVE → IDLE → SCAN → MOVE → MOVE → IDLE → CHECK → MOVE → MOVE → PACK → MOVE → CHECK
Label 0 example (#1):
  MOVE → CHECK → MOVE → MOVE → CHECK → IDLE → MOVE → PACK → IDLE → MOVE → CHECK → MOVE → SCAN → MOVE → MOVE → IDLE

First time-step vector of first sequence: [1. 0. 0. 0. 0.] -> IDLE
Exactly one SCAN & one PACK in all train sequences: True


### A2. Reusable training and evaluation functions

Using **PyTorch**, implement your own code that:

- Converts the provided NumPy arrays to suitable PyTorch tensors (which dtype does `CrossEntropyLoss` expect for the labels?).
- Trains a model on the **training split** using mini-batches, cross-entropy loss and Adam, with seeded shuffling.
- Performs the normal training steps: forward pass, loss, clear old gradients, backward pass and optimizer update.
- Computes **accuracy** without updating the model: use `model.eval()` and `torch.no_grad()`.
- Prints the training loss and the **validation** accuracy after every epoch.

*You will reuse these functions in Parts B–E.*

In [ ]:
# A2 — Reusable pipeline
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 0
HIDDEN_SIZE = 24
BATCH_SIZE = 128
EPOCHS = 10
LR = 0.01

# Inputs -> float32, labels -> int64 (long), as CrossEntropyLoss expects class indices
Xtr, ytr = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long) # training data
Xva, yva = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long) # validation data
Xte, yte = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long) #testing data

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).argmax(dim=1) # returns the highest scoring class index for each input sequence

def accuracy(model, X, y): # accuracy is the fraction of correct predictions
    return (predict(model, X) == y).float().mean().item()

def train_model(model, X, y, X_val, y_val, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=SEED):
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for xb, yb in loader:
            logits = model(xb)            # forward
            loss = loss_fn(logits, yb)    # loss
            opt.zero_grad()               # clear old gradients
            loss.backward()               # backward
            opt.step()                    # update
            total += loss.item() * len(xb); n += len(xb)
        val_acc = accuracy(model, X_val, y_val)
        print(f"Epoch {epoch:2d} | train loss {total / n:.4f} | val acc {val_acc:.4f}")
    return val_acc

## B — Order-blind baseline

Create a small model that first **averages the 16 input vectors over the time dimension** and then applies a linear layer mapping **5 features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy. This model knows the **counts/proportions of events**, but not their order.

**Predict before you run:** can this model learn SCAN-before-PACK? Give a reason based on the *information lost by averaging*, then check your prediction.

In [10]:
# B — Order-blind baseline
class MeanBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 2) # B is the batch size, 5 is the number of input features, and 2 is the number of output classes
    def forward(self, x):              # x: (B, T, 5)
        return self.fc(x.mean(dim=1))  # average over time -> (B, 5) -> (B, 2)

torch.manual_seed(SEED)
baseline = MeanBaseline()
val_base = train_model(baseline, Xtr, ytr, Xva, yva) # train the baseline model and get validation accuracy
print(f"\nBaseline validation accuracy: {val_base:.4f}")

Epoch  1 | train loss 0.6959 | val acc 0.4833
Epoch  2 | train loss 0.6928 | val acc 0.4833
Epoch  3 | train loss 0.6928 | val acc 0.4833
Epoch  4 | train loss 0.6931 | val acc 0.4875
Epoch  5 | train loss 0.6929 | val acc 0.5083
Epoch  6 | train loss 0.6928 | val acc 0.5083
Epoch  7 | train loss 0.6924 | val acc 0.4833
Epoch  8 | train loss 0.6926 | val acc 0.4917
Epoch  9 | train loss 0.6923 | val acc 0.4792
Epoch 10 | train loss 0.6925 | val acc 0.4625

Baseline validation accuracy: 0.4625


## C — Vanilla RNN classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.RNN`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state** as the sequence representation.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy.

**Shape checkpoint:** With batch size `B`, the input is `(B, 16, 5)`; for a one-layer unidirectional RNN, the final hidden-state tensor is `(1, B, 24)`; the classifier output must be `(B, 2)`. **Print these three shapes for one small batch.**

**Defense checkpoint:** What information must the final hidden state carry for this task?

In [ ]:
# C — Vanilla RNN
class RNNClassifier(nn.Module):
    def __init__(self, hidden_size=HIDDEN_SIZE):
        super().__init__() 
        self.rnn = nn.RNN(input_size=5, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2) # takes the final hidden state and produces 2 output values (one for each class)
    def forward(self, x, return_hidden=False): # x is input sequence, 
        _, h_n = self.rnn(x)           # h_n: (1, B, H), process all 16 time steps
        logits = self.fc(h_n[-1])      # (B, H) -> (B, 2)
        return (logits, h_n) if return_hidden else logits

torch.manual_seed(SEED)
rnn = RNNClassifier()

# Shape checkpoint on a small batch
xb = Xtr[:8] # this is a small batch of 8 sequences from the training set
logits, h_n = rnn(xb, return_hidden=True) # 
print("Input:", tuple(xb.shape), "| h_n:", tuple(h_n.shape), "| logits:", tuple(logits.shape), "\n")

val_rnn = train_model(rnn, Xtr, ytr, Xva, yva)
print(f"\nRNN validation accuracy: {val_rnn:.4f}")

Input: (8, 16, 5) | h_n: (1, 8, 24) | logits: (8, 2) 

Epoch  1 | train loss 0.6977 | val acc 0.5792
Epoch  2 | train loss 0.6653 | val acc 0.7292
Epoch  3 | train loss 0.5478 | val acc 0.8583
Epoch  4 | train loss 0.2867 | val acc 0.8958
Epoch  5 | train loss 0.1893 | val acc 0.9375
Epoch  6 | train loss 0.0651 | val acc 0.9667
Epoch  7 | train loss 0.1359 | val acc 0.9667
Epoch  8 | train loss 0.0778 | val acc 0.9750
Epoch  9 | train loss 0.0309 | val acc 0.9583
Epoch 10 | train loss 0.0482 | val acc 0.9875

RNN validation accuracy: 0.9875


## D — LSTM classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.LSTM`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state `h_n`**, not the cell state `c_n`, as the input to the classifier.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with **exactly the same settings as the RNN** and report its **validation** accuracy.

**API checkpoint:** The RNN returns `(outputs, h_n)`, while the LSTM returns `(outputs, (h_n, c_n))`.

**Defense checkpoint:** In simple words, what is the purpose of `c_n`? How is it different from `h_n`? **You do not need to implement the gates from scratch.**

In [7]:
# D — LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.lstm = nn.LSTM(input_size=5, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)
    def forward(self, x):
        _, (h_n, c_n) = self.lstm(x)   # use h_n, not c_n
        return self.fc(h_n[-1])

torch.manual_seed(SEED)
lstm = LSTMClassifier()
val_lstm = train_model(lstm, Xtr, ytr, Xva, yva)
print(f"\nLSTM validation accuracy: {val_lstm:.4f}")

Epoch  1 | train loss 0.6882 | val acc 0.6542
Epoch  2 | train loss 0.6312 | val acc 0.8333
Epoch  3 | train loss 0.3354 | val acc 0.9167
Epoch  4 | train loss 0.1293 | val acc 0.9333
Epoch  5 | train loss 0.0689 | val acc 0.9833
Epoch  6 | train loss 0.0376 | val acc 0.9917
Epoch  7 | train loss 0.0164 | val acc 0.9917
Epoch  8 | train loss 0.0092 | val acc 0.9958
Epoch  9 | train loss 0.0051 | val acc 0.9958
Epoch 10 | train loss 0.0048 | val acc 0.9875

LSTM validation accuracy: 0.9875


## E — Final test evaluation and the reversal experiment

All three models are now trained. Do not change any model or setting from here on.

1. Evaluate the baseline, RNN and LSTM on the **test set, once**. These are your reported test accuracies.
2. Reverse each test sequence **along the time dimension only** (keep its 5 features intact).
3. Check that the number of each type of event is unchanged by reversal.
4. Evaluate all three models on the reversed inputs **against the original `y_test` labels**. Label this as a **diagnostic**, not normal test accuracy.
5. Compare predictions: compute the **flip rate**, the fraction of test sequences whose predicted class changes after reversal.
6. Reason about the **correct** labels for the reversed sequences. (Hint: every sequence contains exactly one SCAN and one PACK.)

**Before you conclude anything:** a model can score close to 0% in step 4. Think about what that number means before you decide whether the model is broken.

**Questions:** Which models respond when the order changes? Why might the order-blind model give the same predictions after reversal?

In [8]:
# E — Test evaluation (once) and reversal experiment
models = {"Order-blind baseline": baseline, "Vanilla RNN": rnn, "LSTM": lstm}
val_accs = {"Order-blind baseline": val_base, "Vanilla RNN": val_rnn, "LSTM": val_lstm}

# 2. Reverse along time dimension only (dim=1); features untouched
Xte_rev = torch.flip(Xte, dims=[1])

# 3. Event counts unchanged
same_counts = torch.equal(Xte.sum(dim=1), Xte_rev.sum(dim=1))
print("Event counts unchanged by reversal:", same_counts)

# 6. Correct labels for reversed sequences: SCAN/PACK order swaps, so label flips
y_rev_true = 1 - yte

print(f"\n{'Model':22s} {'Test acc':>9s} {'Rev vs orig (diag)':>19s} {'Rev vs true rev':>16s} {'Flip rate':>10s}")
results = {}
for name, m in models.items():
    p_orig = predict(m, Xte)
    p_rev = predict(m, Xte_rev)
    test_acc = (p_orig == yte).float().mean().item()
    rev_diag = (p_rev == yte).float().mean().item()
    rev_true = (p_rev == y_rev_true).float().mean().item()
    flip = (p_orig != p_rev).float().mean().item()
    results[name] = (val_accs[name], test_acc, rev_diag, flip)
    print(f"{name:22s} {test_acc:9.4f} {rev_diag:19.4f} {rev_true:16.4f} {flip:10.4f}")

print("\n(Diagnostic only: reversed inputs scored against the ORIGINAL y_test.)")
print("\nMarkdown table for Part F:")
print("| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |")
print("|---|---:|---:|---:|---:|")
for name, (v, t, r, f) in results.items():
    print(f"| {name} | {v:.4f} | {t:.4f} | {r:.4f} | {f:.4f} |")

Event counts unchanged by reversal: True

Model                   Test acc  Rev vs orig (diag)  Rev vs true rev  Flip rate
Order-blind baseline      0.5125              0.5125           0.4875     0.0000
Vanilla RNN               0.9917              0.0125           0.9875     0.9792
LSTM                      1.0000              0.0042           0.9958     0.9958

(Diagnostic only: reversed inputs scored against the ORIGINAL y_test.)

Markdown table for Part F:
| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.4625 | 0.5125 | 0.5125 | 0.0000 |
| Vanilla RNN | 0.9875 | 0.9917 | 0.0125 | 0.9792 |
| LSTM | 0.9875 | 1.0000 | 0.0042 | 0.9958 |


## F — Results and conclusions

### Results table

Values printed by the notebook (Restart & Run All, `SEED = 0`).

| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.4625 | 0.5125 | 0.5125 | 0.0000 |
| Vanilla RNN | 0.9875 | 0.9917 | 0.0125 | 0.9792 |
| LSTM | 0.9875 | 1.0000 | 0.0042 | 0.9958 |

### Short answers (1–3 sentences each)

**1.** Why can a model that averages events over time not reliably distinguish SCAN-before-PACK from PACK-before-SCAN?

**Answer:** Averaging over time only keeps the proportion of each event type and throws away *when* each event happened. Every sequence has exactly one SCAN and one PACK, so the SCAN and PACK features are always 1/16 whatever the label. The only remaining signal is the random IDLE/CHECK/MOVE counts, which tell you nothing about the label, so the baseline stays at chance (≈0.46 val, 0.51 test).

**2.** In `(B, T, 5)`, what are `B` and `T`? What is the shape of the final output logits for `B = 8`? Why?

**Answer:** `B` is the batch size (number of sequences) and `T = 16` is the number of time steps. 5 is the one-hot event size. For `B = 8` the logits are `(8, 2)`: the classifier reads only the final hidden state `h_n[-1]` of shape `(8, 24)`, so the time dimension is gone, and `nn.Linear(24, 2)` produces one score per class for each sequence.

**3.** What is the difference between the hidden state `h_n` and the LSTM cell state `c_n`?

**Answer:** `c_n` is the LSTM's internal long-term memory. It is updated mostly by addition (`c_t = f·c_{t-1} + i·g`), so the forget and input gates control what gets kept or written. `h_n = o·tanh(c_n)` is a gated, squashed view of that memory: the output exposed at each step and the vector we pass to the classifier.

**4.** Which model worked best **in your experiment**? Is the difference between the RNN and the LSTM large enough to matter here? Does this mean LSTM always beats a vanilla RNN? Why or why not?

**Answer:** The LSTM was best on test (1.0000 vs 0.9917 for the RNN), but both had the same validation accuracy (0.9875). The 0.008 test gap is about 2 sequences out of 240, which is within seed-to-seed noise (G1: RNN 0.9875–0.9958, LSTM 0.9875–1.0000), so it does not matter here. LSTM does not always win: with only 16 steps the vanilla RNN does not suffer badly from vanishing gradients. The LSTM's gating mostly helps with long-range dependencies, at the cost of about 4× more parameters.

**5.** After reversal, an order-aware model can score close to 0% against the **original** labels. Is that a failure? Explain what this diagnostic and the flip rate tell you about which models use order.

**Answer:** It is not a failure. Reversing a sequence swaps the order of SCAN and PACK, so its correct label becomes `1 − y`. Scoring ≈0% against the original labels therefore means the RNN (0.0125) and LSTM (0.0042) are predicting the *new* correct label (≈99% against `1 − y`). The flip rates of ≈98–100% show that both recurrent models depend on order. The baseline has a flip rate of 0: the mean over time does not change when you reverse a sequence, so it gives identical predictions, which confirms it is order-blind.

## G — Optional challenges (not graded)

Only if you have finished A–F. None of these affects your 4 points, but they make good defense material.

- **G1. Is the difference real?** Retrain the RNN and LSTM with seeds `0, 1, 2` and compare their **validation** accuracies. Is the RNN–LSTM gap larger than the difference between two seeds of the same model?
- **G2. Confusion matrices.** Write your own 2×2 confusion matrix (rows = true class, columns = predicted class) and print it for each model on the original and the reversed test inputs. What pattern appears for the RNN after reversal, and why does the baseline’s matrix not change?
- **G3. Parameters.** Derive by hand the number of parameters of your RNN and LSTM, check with `sum(p.numel() for p in model.parameters())`, and explain why the LSTM has about 4× more.
- **G4. Longer sequences (no code).** In Homework 7 you computed how an early input’s influence shrinks as $w^{T-1}$. Using this, explain why a vanilla RNN would struggle if SCAN and PACK could be 50 steps before the end. How does the LSTM update $c_t = f \cdot c_{t-1} + i \cdot g$ avoid this when the forget gate $f$ is close to 1?

*Use the empty cell below if you attempt G1–G3, and write G4 as text.*

In [9]:
# G1 — seed variation
for s in [0, 1, 2]:
    for name, cls in [("RNN", RNNClassifier), ("LSTM", LSTMClassifier)]:
        torch.manual_seed(s)
        m = cls()
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            v = train_model(m, Xtr, ytr, Xva, yva, seed=s)
        print(f"seed {s} | {name:4s} val acc {v:.4f}")

# G2 — confusion matrices
def confusion(y_true, y_pred):
    cm = torch.zeros(2, 2, dtype=torch.long)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

for name, m in models.items():
    print(f"\n{name}\n original:\n{confusion(yte, predict(m, Xte)).numpy()}"
          f"\n reversed (vs original labels):\n{confusion(yte, predict(m, Xte_rev)).numpy()}")

# G3 — parameter counts
# RNN : W_ih 24*5 + W_hh 24*24 + b_ih 24 + b_hh 24 = 744 ; head 24*2+2 = 50 -> 794
# LSTM: 4*(24*5 + 24*24 + 24 + 24) = 2976 ; head 50 -> 3026
for name, m in [("RNN", rnn), ("LSTM", lstm)]:
    print(f"{name} params: {sum(p.numel() for p in m.parameters())}")

seed 0 | RNN  val acc 0.9875
seed 0 | LSTM val acc 0.9875
seed 1 | RNN  val acc 0.9958
seed 1 | LSTM val acc 0.9875
seed 2 | RNN  val acc 0.9875
seed 2 | LSTM val acc 1.0000

Order-blind baseline
 original:
[[42 78]
 [39 81]]
 reversed (vs original labels):
[[42 78]
 [39 81]]

Vanilla RNN
 original:
[[118   2]
 [  0 120]]
 reversed (vs original labels):
[[  0 120]
 [117   3]]

LSTM
 original:
[[120   0]
 [  0 120]]
 reversed (vs original labels):
[[  0 120]
 [119   1]]
RNN params: 794
LSTM params: 3026


### G1–G4 notes

- **G1:** The RNN–LSTM validation gap (0–0.0125) is about the same size as the variation between seeds of one model, so it is not a real difference on this task.
- **G2:** After reversal, the RNN and LSTM confusion matrices become almost purely anti-diagonal: every class is predicted as the other one, which is exactly the correct label for the reversed sequence. The baseline's matrix is identical because the mean over time is unchanged by reversal.
- **G3:** RNN: `24·5 + 24·24 + 24 + 24 = 744` recurrent + `24·2 + 2 = 50` head = **794**. LSTM: the same blocks for each of 4 gates (input, forget, cell candidate, output), so `4·744 = 2976` + 50 = **3026**. That is about 4× because each gate has its own weights.
- **G4:** In a vanilla RNN, the gradient from an early input to the final state is scaled by roughly `w^{T-1}`. With `|w| < 1` and SCAN/PACK 50 steps before the end, this factor is near zero (e.g. 0.9^49 ≈ 0.006), so the model can neither remember nor learn from those events. In the LSTM, `c_t = f·c_{t-1} + i·g` has an additive path where `∂c_t/∂c_{t-1} = f`. With `f ≈ 1`, information and gradients pass through many steps almost unchanged (the "constant error carousel").

### Team members and contributions

- **Student 1** — Full name / student ID / contribution: …
- **Student 2** — Full name / student ID / contribution: …
- **Student 3** — Full name / student ID / contribution: …

### Before submitting

- **Kernel → Restart & Run All** finishes without errors, and the numbers in your Part F table match the printed outputs.
- The test set is evaluated only in Part E.
- Each team member can explain any chosen code line during the **individual oral defense**.

**Submission:** One completed `.ipynb` per team via Microsoft Teams. No separate report required.